# Rolling ablation cho trend và so sánh calibration

Notebook này thực hiện hai bước nghiên cứu riêng: (1) kiểm tra feature trend qua rolling folds của train, và (2) so sánh raw, Platt và isotonic với tập fit calibration và tập so sánh tách biệt theo thời gian. Không đọc test, không ghi đè pipeline POC đã chốt. Feature và quy trình đã được thảo luận qua validation trước đây; kết quả không phải xác nhận độc lập.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.special import logit
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import brier_score_loss,log_loss,roc_auc_score
from IPython.display import display
ROOT=Path.cwd()
if ROOT.name == 'research' and ROOT.parent.name == 'notebooks': ROOT = ROOT.parent
if ROOT.name=='notebooks':ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/'src'))
if (ROOT/'.deps').is_dir():sys.path.insert(0,str(ROOT/'.deps'))
from propensity_v2.prepared_data import load_prepared_split
from lightgbm import LGBMClassifier
OUT=ROOT/'artifacts'/'research'/'trend_calibration_research'
train,features=load_prepared_split(ROOT/'artifacts'/'data_preparation','train')
print('Train:',len(train),'| cutoff:',train.snapshot_day.nunique())

Train: 129834 | cutoff: 52


## 1. Trend không chồng lấn

So sánh trung bình/ngày của 7 ngày gần nhất với 23 ngày trước đó; so sánh 90 ngày gần nhất với 90 ngày trước đó. Với household chưa đủ 30 hoặc 180 ngày từ lần đầu quan sát, giữ trend tương ứng là thiếu và thêm cờ đủ lịch sử. Đủ thời gian từ lần đầu không xác nhận nguồn đầy đủ. `log1p` là sai khác trên mức trung bình/ngày.

In [2]:
def add_trend(frame):
    out=frame.copy()
    out['trend_has_30d_history']=(out.observed_tenure_days+1>=30).astype(int)
    out['trend_has_180d_history']=(out.observed_tenure_days+1>=180).astype(int)
    cols=['trend_has_30d_history','trend_has_180d_history']
    for metric in ['baskets','sales']:
        recent=out[f'{metric}_7d']/7;prior=(out[f'{metric}_30d']-out[f'{metric}_7d'])/23
        if (prior < -1e-8).any():raise ValueError('Khoảng trước 23 ngày có tổng âm; cần kiểm tra')
        col=f'{metric}_trend_7_vs_previous23'
        out[col]=np.where(out.trend_has_30d_history.eq(1),np.log1p(recent)-np.log1p(prior.clip(lower=0)),np.nan);cols.append(col)
    for metric in ['baskets','sales','active_days']:
        recent=out[f'{metric}_90d']/90;prior=(out[f'{metric}_180d']-out[f'{metric}_90d'])/90
        if (prior < -1e-8).any():raise ValueError('Khoảng trước 90 ngày có tổng âm; cần kiểm tra')
        col=f'{metric}_trend_90_vs_previous90'
        out[col]=np.where(out.trend_has_180d_history.eq(1),np.log1p(recent)-np.log1p(prior.clip(lower=0)),np.nan);cols.append(col)
    return out,cols
data,trend_cols=add_trend(train)
print('Số feature trend và cờ:',len(trend_cols))

Số feature trend và cờ: 7


## 2. Rolling ablation

Sử dụng bốn fold có 6 cutoff đánh giá, train mở rộng theo thời gian và bỏ một cutoff embargo. So sánh bộ 18 feature với 18 + trend cho cùng model và cutoff. Brier/log loss càng thấp càng tốt; ROC-AUC/lift càng cao càng tốt. Không chọn/xóa feature tự động theo một lần importance âm.

In [3]:
def make_model(name):
    if name=='Logistic Regression':return make_pipeline(SimpleImputer(strategy='median',keep_empty_features=True),StandardScaler(),LogisticRegression(max_iter=2000,random_state=42))
    return LGBMClassifier(n_estimators=200,learning_rate=.05,num_leaves=15,min_child_samples=100,random_state=42,verbosity=-1,n_jobs=2)
def lift(y,p):
    y=np.asarray(y);p=np.asarray(p);k=int(np.ceil(.1*len(y)));t=np.partition(p,len(p)-k)[len(p)-k]
    high=p>t;ties=p==t
    return ((y[high].sum()+(k-high.sum())*y[ties].mean())/k)/y.mean() if y.mean() else np.nan
def metrics(frame,p):
    y=frame.target_any_basket.to_numpy()
    weekly=[lift(y[ix],p[ix]) for ix in frame.groupby('snapshot_day').indices.values()]
    return {'brier':brier_score_loss(y,p),'log_loss':log_loss(y,p,labels=[0,1]),
            'roc_auc':roc_auc_score(y,p) if np.unique(y).size==2 else np.nan,'weekly_lift':np.nanmean(weekly)}
days=sorted(data.snapshot_day.unique());fold_rows=[];fold_info=[]
for fold,start in enumerate([20,28,36,44],1):
    train_days=days[:start-1];eval_days=days[start:start+6]
    assert len(eval_days)==6 and max(train_days)+7<min(eval_days)
    tr=data.loc[data.snapshot_day.isin(train_days)].reset_index(drop=True)
    ev=data.loc[data.snapshot_day.isin(eval_days)].reset_index(drop=True)
    fold_info.append({'fold':fold,'train_end':max(train_days),'embargo':days[start-1],
                      'eval_start':min(eval_days),'eval_end':max(eval_days),'train_rows':len(tr),'eval_rows':len(ev)})
    for name in ['Logistic Regression','LightGBM']:
        for variant,extra in [('18 feature',[]),('18 + trend',trend_cols)]:
            cols=features+extra;model=make_model(name);model.fit(tr[cols],tr.target_any_basket)
            p=model.predict_proba(ev[cols])[:,1]
            fold_rows.append({'fold':fold,'model':name,'variant':variant,**metrics(ev,p)})
fold_results=pd.DataFrame(fold_rows)
base=fold_results.loc[fold_results.variant.eq('18 feature')].drop(columns='variant').rename(columns={m:'base_'+m for m in ['brier','log_loss','roc_auc','weekly_lift']})
diff=fold_results.loc[fold_results.variant.eq('18 + trend')].merge(base,on=['fold','model'],validate='one_to_one')
for m in ['brier','log_loss','roc_auc','weekly_lift']:diff['delta_'+m]=diff[m]-diff['base_'+m]
display(pd.DataFrame(fold_info));display(diff[['fold','model','delta_brier','delta_log_loss','delta_roc_auc','delta_weekly_lift']])

,fold,train_end,embargo,eval_start,eval_end,train_rows,eval_rows
0,1,306,313,320,355,47418,14982
1,2,362,369,376,411,67394,14982
2,3,418,425,432,467,87370,14987
3,4,474,481,488,523,107352,14988


,fold,model,delta_brier,delta_log_loss,delta_roc_auc,delta_weekly_lift
0,1,Logistic Regression,-0.000045,-0.000134,0.000094,-0.000035
1,1,LightGBM,0.000193,0.000502,-0.000369,0.001309
2,2,Logistic Regression,-0.000240,-0.000530,0.000201,0.001289
3,2,LightGBM,-0.000197,-0.000479,0.000325,-0.002542
4,3,Logistic Regression,0.000040,0.000189,0.000003,0.002596
5,3,LightGBM,-0.000272,-0.000675,0.000461,0.007452
6,4,Logistic Regression,-0.000103,-0.000224,0.000339,0.000028
7,4,LightGBM,0.000060,0.000156,-0.000115,-0.007582


## 3. So sánh raw, Platt và isotonic trên các giai đoạn khác nhau

Để khảo sát riêng calibration, dùng **LightGBM 18 feature**: fit model đến cutoff 439; fit Platt và isotonic trên 453–481; so sánh cả ba trên 495–537. Cutoff 446 và 488 được embargo. Bảng là so sánh ngoài mẫu đối với các calibrator, nhưng không phải đánh giá độc lập cho lựa chọn cuối vì nó đang được dùng thăm dò.

Isotonic có thể làm nhiều điểm trùng nhau; log loss dùng giá trị trong [0,1] và ROC-AUC có thể thay đổi do đồng hạng. Không tự động thay pipeline đã chốt; nếu quyết định đổi calibration thì cần tập đánh giá chưa dùng để lựa chọn.

In [4]:
fit=data.snapshot_day.between(180,439);cal=data.snapshot_day.between(453,481);evaluate=data.snapshot_day.between(495,537)
assert data.loc[fit,'snapshot_day'].max()+7<data.loc[cal,'snapshot_day'].min()
assert data.loc[cal,'snapshot_day'].max()+7<data.loc[evaluate,'snapshot_day'].min()
model=make_model('LightGBM');model.fit(data.loc[fit,features],data.loc[fit,'target_any_basket'])
cal_prob=model.predict_proba(data.loc[cal,features])[:,1]
eval_frame=data.loc[evaluate].reset_index(drop=True);raw_prob=model.predict_proba(eval_frame[features])[:,1]
def as_logit(p):return logit(np.clip(p,1e-6,1-1e-6)).reshape(-1,1)
platt=LogisticRegression(C=1e6,max_iter=1000,random_state=42).fit(as_logit(cal_prob),data.loc[cal,'target_any_basket'])
iso=IsotonicRegression(out_of_bounds='clip').fit(cal_prob,data.loc[cal,'target_any_basket'])
probs={'raw':raw_prob,'platt':platt.predict_proba(as_logit(raw_prob))[:,1],'isotonic':iso.predict(raw_prob)}
calibration=pd.DataFrame([{'method':name,**metrics(eval_frame,p),'unique_scores':len(np.unique(p))} for name,p in probs.items()])
display(calibration)

,method,brier,log_loss,roc_auc,weekly_lift,unique_scores
0,raw,0.164990,0.496930,0.835979,1.808266,16191
1,platt,0.164956,0.496889,0.835979,1.808266,16191
2,isotonic,0.165163,0.497544,0.835358,1.807320,128


**Kết quả:** Trend không cải thiện Brier nhất quán qua bốn fold ở cả hai model: LightGBM tốt hơn ở 2/4 fold và Logistic ở 3/4 fold. Trên giai đoạn chọn calibration 495–537, isotonic có Brier 0,16516 và log loss 0,49754, cao hơn raw 0,16499 và 0,49693; Platt khác raw rất ít. Đây là so sánh thăm dò cho LightGBM 18 feature, chưa thay đổi pipeline đã chốt.


## 4. Giới hạn và bàn giao

Không coi chênh lệch Brier rất nhỏ là cải thiện chắc chắn hay “calibration trade-off” đã chứng minh: Brier còn phản ánh khả năng phân biệt. Không gán nguyên nhân lift giảm cho feature tương quan; hệ số tương quan đơn cũng không đủ khẳng định dư thừa. Unit economics thật và tác động can thiệp không có trong dataset. Các thí nghiệm chưa tạo mô hình production mới.

In [5]:
OUT.mkdir(parents=True,exist_ok=True)
fold_results.to_csv(OUT/'rolling_trend_metrics.csv',index=False,encoding='utf-8-sig')
diff.to_csv(OUT/'rolling_trend_differences.csv',index=False,encoding='utf-8-sig')
calibration.to_csv(OUT/'calibration_comparison_train_holdout.csv',index=False,encoding='utf-8-sig')
print('Đã lưu:',OUT)

Đã lưu: artifacts\trend_calibration_research
